# Gym Booking Automation

* Log in automatically

* Book specific gym classes

* Handle waitlists

* Deal with network errors like a pro

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
#we have to wait until the page is charge so 
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

In [2]:
env_file = Path.cwd().parent / '.env'
load_dotenv(env_file)
PASSWORD = os.getenv('GYM_PASSWORD')
ACCOUNT = os.getenv('GYM_ACCOUNT')
GYM_URL = "https://appbrewery.github.io/gym/"

In [3]:
#stay open using the Chrome option:
chrome_options = webdriver.ChromeOptions()
chrome_options.add_experimental_option("detach", True)
#Create a folder for the Chrome Profile selenium AND will use it every time
user_data_dir = os.path.join(os.getcwd(), "chrome_profile")
#use the directory specified to store a "profile".
chrome_options.add_argument(f"--user-data-dir={user_data_dir}")

driver = webdriver.Chrome(options=chrome_options)
driver.get(GYM_URL)

In [4]:
#this will retries instead of immediately raising an exception
wait = WebDriverWait(
    driver,
    15, #wait 15 seconds
    poll_frequency=0.2, #wait 0.2 seconds
    ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
)

# Wait until the language button is clickable
wait.until(
    #we get the element that we are looking
    EC.element_to_be_clickable((By.ID, 'login-button'))
#and then make click    
).click()

In [5]:
#now fill the inputs with our data
email_input = driver.find_element(By.ID, value='email-input')
email_input.send_keys(ACCOUNT)
password_input = driver.find_element(By.ID, value='password-input')
password_input.send_keys(PASSWORD)

#AND SEND IT
driver.find_element(By.ID, value='submit-button').click()

# Now once we access the instructions says

**Book Your First Class - Tuesday at 6pm**

In [ ]:
#find the buttons
#Lets see the condition on the value
# first we are looking for all buttons then []
# means the ID starts with book-button-.

list_buttons = driver.find_elements(By.CSS_SELECTOR, value="button[id^='book-button-']")
for i in range(0,5):
    print("ID:", list_buttons[i].get_attribute("id"))
    print("Text:", list_buttons[i].text.strip())
    print("Tag:", list_buttons[i].tag_name)
    print("Aria busy:", list_buttons[i].get_attribute("aria-busy"))
    print("Aria disabled:", list_buttons[i].get_attribute("aria-disabled"))
    print("Enabled:", list_buttons[i].is_enabled())
    print("--------------------")

# The problem (for specific classes) is that we have a lot of buttons 
# we can see all of them until we reach the date we wanted, but
# we do not know what class will be (it could be any and we do not know that) 

ID: book-button-yoga-2026-11-06-1700
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-spin-2026-11-06-1800
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-hiit-2026-11-06-1900
Text: Join Waitlist
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-yoga-2026-11-07-0700
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-spin-2026-11-07-0800
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------


### (Just for the future this method of getting all the buttons will help us to book more clasess, but we need to figure if they are enable to click)

In [ ]:
for button in list_buttons:
    #we join all the text and ignore uppercase or lowercase
    button_text = " ".join(button.text.split()).casefold()
    #and compare first if it says book class, and can be click it
    if button_text == "book class" and button.is_enabled():
        #then we click it to reserve it
        #(for now we just print the id of the ones that are availables)
        print("Available:", button.get_attribute("id"))

Available: book-button-yoga-2026-11-06-1700
Available: book-button-spin-2026-11-06-1800
Available: book-button-yoga-2026-11-07-0700
Available: book-button-spin-2026-11-07-0800
Available: book-button-yoga-2026-11-07-1700
Available: book-button-spin-2026-11-07-1800
Available: book-button-hiit-2026-11-07-1900
Available: book-button-yoga-2026-11-08-0700
Available: book-button-hiit-2026-11-08-0900
Available: book-button-yoga-2026-11-08-1700
Available: book-button-spin-2026-11-08-1800
Available: book-button-hiit-2026-11-08-1900
Available: book-button-yoga-2026-11-09-0700
Available: book-button-hiit-2026-11-09-0900
Available: book-button-yoga-2026-11-09-1700
Available: book-button-hiit-2026-11-09-1900
Available: book-button-yoga-2026-11-10-0700
Available: book-button-spin-2026-11-10-0800
Available: book-button-hiit-2026-11-10-0900
Available: book-button-yoga-2026-11-10-1700
Available: book-button-spin-2026-11-10-1800
Available: book-button-hiit-2026-11-10-1900
Available: book-button-yoga-2026

# For Tuesday (specific class) First we need to get the date

In [38]:
from datetime import date, timedelta, datetime

In [8]:
#calculate next tuesday
today = date.today()

# tuesday = 1 (Monday is 0)
days_until_tuesday = (1 - today.weekday()) % 7

# If today is tuesday, select next tuesday
if days_until_tuesday == 0:
    days_until_tuesday = 7

#now we have the days, we need the date, so we are going to add the days
#for the right one
next_date = today + timedelta(days=days_until_tuesday)
#and we convert the date into a string
date_string = next_date.strftime("%Y-%m-%d")

print(date_string)

2026-11-10


# Then updated the search by only one element with that id

In [25]:
# Now we updated the condition
# For the second part means the ID contains the selected date. (plus the hour that in this case we can put it)
button = driver.find_element(By.CSS_SELECTOR, 
                             value=f"button[id^='book-button-'][id*='{date_string}-1800']")

print("ID:", button.get_attribute("id"))
print("Text:", button.text.strip())
print("Tag:", button.tag_name)
print("Aria busy:", button.get_attribute("aria-busy"))
print("Aria disabled:", button.get_attribute("aria-disabled"))
print("Enabled:", button.is_enabled())
print("--------------------")

ID: book-button-spin-2026-11-10-1800
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------


In [26]:
# we do click
button.click()

In [29]:
#now the result
button = driver.find_element(By.CSS_SELECTOR, 
                             value=f"button[id^='book-button-'][id*='{date_string}-1800']")
#as we see it is the class booked and the button is disable 
# that means we can not do click again
print("ID:", button.get_attribute("id"))
print("Text:", button.text.strip())
print("Tag:", button.tag_name)
print("Aria busy:", button.get_attribute("aria-busy"))
print("Aria disabled:", button.get_attribute("aria-disabled"))
print("Enabled:", button.is_enabled())
print("--------------------")

ID: book-button-spin-2026-11-10-1800
Text: Booked
Tag: button
Aria busy: None
Aria disabled: None
Enabled: False
--------------------


# As the instruction says we need now to **Check if a class is already booked** but that already was done, now we only need to updated our condition

Instaed of printing the ones that are available we are going to print

* Check if a class is already booked (button reads "Booked")
  * Example:
    * Already booked: Spin Class on Tue, Aug 12
* Check if you're on the waitlist (button reads "Waitlisted")
  * Example:
    * Already on waitlist: HIIT Class on Tue, Aug 12
* Join the waitlist if the class is full (button says "Join Waitlist")
  * Example:
    * Joined waitlist for: Yoga Class on Tue, Aug 12

In [46]:
# all the data is on
print(button.get_attribute("id"))
#book-button-yoga-2026-11-06-1700

#with that we can split it to get the class, year, month and day
all_data_button = button.get_attribute("id").split('-')
print(all_data_button)
#now the data
schedule_class = all_data_button[2]
print(schedule_class)

schedule_year = int(all_data_button[3])
print(schedule_year)

schedule_day = int(all_data_button[5])
print(schedule_day)

#what about the month, we wanted text not number
schedule_month = int(all_data_button[4])
#we need to pass like this (year, month, day).(%B) name of month
#it is a lot (and the data need to be on integer)
print(datetime(schedule_year, schedule_month, schedule_day).strftime("%B"))

book-button-spin-2026-11-10-1800
['book', 'button', 'spin', '2026', '11', '10', '1800']
spin
2026
10
November


#Well for fortune we can look for parent
'''
Child =  card element

Parent = the element directly wrapping the card (e.g., a div)

Grandparent = the wrapper around the parent one level up


Our structure is like this

< div id="day-group-today-(fri,-nov-6)">

    < h2 id="day-title-today-(fri,-nov-6)">

        Today (Fri, Nov 6)

    </ h2>

    <!-- Other elements and divs -->

    < button id="book-button-yoga-2026-11-06-1700">

        Book class

    </ button>

</ div>

So lets se how we do it
'''

In [47]:
day_group = button.find_element(
    By.XPATH,
    "ancestor::div[starts-with(@id, 'day-group-')][1]"
)
# ancestor::div searches the parent elements and their parents.
# starts-with(@id, 'day-group-') finds the div whose ID starts with day-group-.
# [1] selects the closest matching ancestor.

In [ ]:
#witht that now we can find the heading
day_title = day_group.find_element(
    By.CSS_SELECTOR, "h2"
).text.strip()
#and that is the date (for each button we can have it like that)
print(day_title)

Tue, Nov 10


# We can keep how we get the class name, the date is going to be like this so lets do it for all the buttons but we are going to do it on a function

In [50]:
#libraries
from pathlib import Path
from dotenv import load_dotenv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
#we have to wait until the page is charge so 
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
from datetime import date, timedelta 

In [51]:
env_file = Path.cwd().parent / '.env'
load_dotenv(env_file)
PASSWORD = os.getenv('GYM_PASSWORD')
ACCOUNT = os.getenv('GYM_ACCOUNT')
GYM_URL = "https://appbrewery.github.io/gym/"

In [90]:
def chrome_configuration():
    #stay open using the Chrome option:
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_experimental_option("detach", True)
    #Create a folder for the Chrome Profile selenium AND will use it every time
    user_data_dir = os.path.join(os.getcwd(), "chrome_profile")
    #use the directory specified to store a "profile".
    chrome_options.add_argument(f"--user-data-dir={user_data_dir}")
    return chrome_options

In [91]:
def make_connection(chrome_options):
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(GYM_URL)
    return driver

In [92]:
def log_in(wait):  
    # Wait until the language button is clickable
    wait.until(
        #we get the element that we are looking
        EC.element_to_be_clickable((By.ID, 'login-button'))
    #and then make click    
    ).click()
    
    #second wait
    # Wait until the email input is visible
    email_input = wait.until(
        EC.visibility_of_element_located((By.ID, 'email-input'))
    )
    email_input.send_keys(ACCOUNT)

    # Wait until the password input is visible
    password_input = wait.until(
        EC.visibility_of_element_located((By.ID, 'password-input'))
    )
    password_input.send_keys(PASSWORD)

    # Wait until the submit button is clickable
    wait.until(
        EC.element_to_be_clickable((By.ID, 'submit-button'))
    ).click()

In [93]:
def find_all_buttons(wait):
    #we will wait until the elements (all the buttons) are displayed (loaded)
    list_buttons = wait.until(
        EC.visibility_of_all_elements_located((
        By.CSS_SELECTOR, 
        "button[id^='book-button-']"
        ))
    )
    return list_buttons

In [94]:
def get_date_title(button):
    # Get date from the ancestor
    day_group = button.find_element(
        By.XPATH,
        "ancestor::div[starts-with(@id, 'day-group-')][1]"
    )

    day_title = day_group.find_element(
        By.CSS_SELECTOR, "h2"
    ).text.strip()
    
    return day_title

In [ ]:
def check_classes(all_buttons, class_names):
    for button in all_buttons:
        button_text = " ".join(button.text.split()).casefold()
        #compare first if the class is other case except for book it
        if button_text != "Book Class":
            #then we get the date and class name
            date_label = get_date_title(button)
            schedule_class = button.get_attribute("id").split('-')[2]
            #if the class name is not on the dictionary, then we put the class on title case
            class_name = class_names.get(schedule_class, schedule_class.title())
            #now we just compare
            if button_text == "booked":
                print(f"Already booked: {class_name} Class on {date_label}")
            elif button_text == "waitlisted":
                print(f"Already on waitlist: {class_name} Class on {date_label}")
            #Join the waitlist (first we do click and the we can show the data)
            elif button_text == "join waitlist":
                button.click()
                print(f"Clicked Join Waitlist for: {class_name} Class on {date_label}")
            

In [101]:
def main():
    #make a dictionary to print the classes
    class_names = {
        "yoga": "Yoga",
        "spin": "Spin",
        "hiit": "HIIT"
    }

    #initialize chrome page with the options
    chrome_options = chrome_configuration()
    driver = make_connection(chrome_options)

    #make a waiter 
    wait = WebDriverWait(
        driver,
        15, #wait 15 seconds
        poll_frequency=0.2, #wait 0.2 seconds
        ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
    )

    #log in
    log_in(wait)

    #the find_all_buttons
    print('Checking the classes...')
    all_buttons = find_all_buttons(wait)

    print('Printnig the results for the classes')
    #check the classes that are not available
    check_classes(all_buttons, class_names)

In [100]:
main()

Checking the classes...
Printnig the results for the classes
Clicked Join Waitlist for: <built-in method title of str object at 0x7f1f6090a6d0> Class on Today (Fri, Nov 6)
Clicked Join Waitlist for: <built-in method title of str object at 0x7f1f617b92c0> Class on Tomorrow (Sat, Nov 7)
Clicked Join Waitlist for: Spin Class on Sun, Nov 8
Clicked Join Waitlist for: Spin Class on Mon, Nov 9
Clicked Join Waitlist for: Spin Class on Mon, Nov 9
Already booked: Spin Class on Tue, Nov 10
Clicked Join Waitlist for: Spin Class on Thu, Nov 12
